# ⭐ Day 74: Policy Gradient Methods & Actor-Critic Algorithms
From Value-Based to Policy-Based Reinforcement Learning

**Python & AI Learning Path — Day 74 of 369** 🧠📈

## 🎯 Introduction

Welcome to Day 74 of your 369-day Python & AI Learning Path! Today marks a pivotal transition in our reinforcement learning journey. Over the past several days, we've mastered value-based methods — from the foundational Q-Learning to the deep and powerful Deep Q-Networks (DQN). These algorithms learn to estimate the value of state-action pairs and derive policies implicitly through value maximization. While elegant and effective for discrete action spaces, they face significant challenges when confronted with continuous or high-dimensional action spaces.

**Policy gradient methods** represent a paradigm shift: instead of learning *values* and deriving policies, we directly parameterize and optimize the policy itself. Imagine the difference between learning a map of a city (value-based) versus learning to drive through it instinctively (policy-based). The latter often proves more natural for complex, continuous control tasks where discretizing actions becomes computationally prohibitive or loses critical nuance.

The true power of modern reinforcement learning emerges when we combine the best of both worlds. **Actor-Critic algorithms** marry the direct policy optimization of policy gradients with the variance-reduction benefits of value estimation. The "Actor" learns *what to do* (the policy), while the "Critic" learns *how good it is* (the value function). This symbiotic relationship has produced some of the most successful RL algorithms in existence, including A3C, PPO, and SAC — the engines behind breakthroughs from game-playing AI to robotic control.

By the end of today, you'll understand the mathematical foundations of the Policy Gradient Theorem, implement the classic REINFORCE algorithm from scratch, build a simple Actor-Critic agent, and gain intuition for why these methods dominate continuous control benchmarks. Let's dive into the policy-based revolution! 🚀

## 📚 Table of Contents

1. [Limitations of Value-Based Methods and the Rise of Policy-Based RL](#1-limitations-of-value-based-methods-and-the-rise-of-policy-based-rl)
2. [Policy Gradient Theorem – The Mathematical Foundation](#2-policy-gradient-theorem--the-mathematical-foundation)
3. [REINFORCE Algorithm (Monte Carlo Policy Gradient)](#3-reinforce-algorithm-monte-carlo-policy-gradient)
4. [Actor-Critic Methods – Combining Value and Policy Learning](#4-actor-critic-methods--combining-value-and-policy-learning)
5. [Advantage Actor-Critic (A2C) and Asynchronous Advantage Actor-Critic (A3C)](#5-advantage-actor-critic-a2c-and-asynchronous-advantage-actor-critic-a3c)
6. [Implementing a Simple Policy Gradient (REINFORCE) from Scratch](#6-implementing-a-simple-policy-gradient-reinforce-from-scratch)
7. [Training on CartPole or a Continuous Control Environment](#7-training-on-cartpole-or-a-continuous-control-environment)
8. [Visualizing Training Progress and Learned Policy](#8-visualizing-training-progress-and-learned-policy)
9. [Pros, Cons, and When to Use Policy Gradient vs DQN](#9-pros-cons-and-when-to-use-policy-gradient-vs-dqn)
10. [Modern Actor-Critic Algorithms (PPO, SAC – brief overview)](#10-modern-actor-critic-algorithms-ppo-sac--brief-overview)
11. [🛠️ Hands-On Exercises](#-hands-on-exercises)
12. [Solutions (Review After Attempting)](#solutions-review-after-attempting)

## 1. Limitations of Value-Based Methods and the Rise of Policy-Based RL

Value-based methods like Q-Learning and DQN have achieved remarkable success, but they carry inherent limitations that motivate policy-based approaches:

### 🔴 Key Limitations of Value-Based Methods

| Limitation | Description | Why It Matters |
|------------|-------------|--------------|
| **Discrete Action Spaces** | Q-tables and argmax operations require finite actions | Cannot directly handle continuous motor controls |
| **Maximization Bias** | Overestimation of Q-values leads to suboptimal policies | Training instability and poor convergence |
| **Deterministic Policies** | Greedy action selection cannot represent stochastic strategies | Misses optimal mixed strategies in games |
| **Exploration Decoupling** | Exploration (ε-greedy) is separate from policy learning | Inefficient exploration in complex spaces |

### 🟢 Why Policy-Based Methods?

- **Direct Optimization**: Optimize the objective we truly care about — expected cumulative reward
- **Continuous Actions**: Naturally handle continuous action spaces via probability distributions
- **Stochastic Policies**: Can learn probabilistic action selections, crucial for partially observable environments
- **Smooth Updates**: Policy changes are typically smooth and differentiable, enabling stable gradient-based learning
- **Better Convergence**: Guaranteed convergence to at least a local optimum under mild conditions

## 2. Policy Gradient Theorem – The Mathematical Foundation

The **Policy Gradient Theorem** provides the mathematical justification for directly optimizing policies via gradient ascent. This is the cornerstone of all policy gradient methods.

### 🧮 The Objective Function

We define the performance objective as the expected return under policy π parameterized by θ:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)] = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right]$$

Where τ represents a trajectory (sequence of states, actions, and rewards).

### 🧮 The Policy Gradient Theorem

The theorem states that the gradient of the objective with respect to policy parameters is:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot G_t\right]$$

Where:
- **$\pi_\theta(a_t \| s_t)$** is the probability of taking action $a_t$ in state $s_t$ under policy parameters θ
- **$\nabla_\theta \log \pi_\theta(a_t \| s_t)$** is the **score function** (or log-likelihood gradient)
- **$G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$** is the **return** (total discounted reward from time t)

### 💡 Intuition

The gradient tells us: *"Increase the probability of actions that led to high returns, and decrease the probability of actions that led to low returns."* The log-probability gradient acts as a direction, while the return $G_t$ acts as a magnitude and sign — scaling updates by how good or bad the outcome was.

### ⚠️ The Variance Problem

Monte Carlo estimates of $G_t$ have high variance. This motivates:
- **Baselines**: Subtract a baseline (like state value $V(s_t)$) to reduce variance without changing the expected gradient
- **Advantage Functions**: Use $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$ instead of raw returns

## 3. REINFORCE Algorithm (Monte Carlo Policy Gradient)

**REINFORCE** (Williams, 1992) is the simplest policy gradient algorithm. It uses Monte Carlo sampling to estimate returns and performs gradient ascent on the policy parameters.

### 📝 Algorithm Pseudocode

```
Initialize policy parameters θ
for each episode:
    Generate trajectory τ = (s₀, a₀, r₀, s₁, a₁, r₁, ..., s_T) following π_θ
    for t = 0 to T:
        Calculate return G_t = Σ γ^(k-t) * r_k  (from t to end)
        Update: θ ← θ + α * ∇_θ log π_θ(a_t | s_t) * G_t
```

### 🔑 Key Characteristics
- **On-policy**: Must collect fresh trajectories using the current policy
- **Monte Carlo**: Uses complete episode returns (no bootstrapping)
- **High Variance**: Raw returns can vary wildly between episodes
- **Simple**: No value function needed — pure policy optimization

## 4. Actor-Critic Methods – Combining Value and Policy Learning

Actor-Critic architectures address REINFORCE's high variance by introducing a **Critic** — a learned value function that bootstraps return estimates.

### 🎭 The Two Roles

| Component | Role | Learns | Reduces |
|-----------|------|--------|---------|
| **Actor** | The Policy | $\pi_\theta(a \| s)$ — what action to take | Directly optimizes objective |
| **Critic** | The Value Function | $V_w(s)$ or $Q_w(s,a)$ — how good is this state/action? | Variance of policy gradient estimates |

### 🧮 Actor-Critic Policy Gradient

Instead of using the Monte Carlo return $G_t$, we use the **TD error** (or advantage) as a lower-variance estimate:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[\nabla_\theta \log \pi_\theta(a_t \| s_t) \cdot \delta_t\right]$$

Where the TD error is:
$$\delta_t = r_t + \gamma V_w(s_{t+1}) - V_w(s_t)$$

This is an unbiased estimate of the advantage function and dramatically reduces variance compared to raw returns.

### 🔄 Update Rules
- **Critic Update** (minimize MSE): $w \leftarrow w + \beta \cdot \delta_t \cdot \nabla_w V_w(s_t)$
- **Actor Update** (maximize performance): $\theta \leftarrow \theta + \alpha \cdot \nabla_\theta \log \pi_\theta(a_t \| s_t) \cdot \delta_t$

## 5. Advantage Actor-Critic (A2C) and Asynchronous Advantage Actor-Critic (A3C)

### 🏗️ A2C: Synchronous Advantage Actor-Critic

A2C explicitly computes the **advantage function** to further reduce variance:

$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t) \approx G_t - V(s_t)$$

Or using n-step returns:
$$A(s_t, a_t) = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n}) - V(s_t)$$

**Benefits**:
- Centers the updates around zero (good actions → positive advantage, bad → negative)
- Much lower variance than REINFORCE
- More stable training

### 🌐 A3C: Asynchronous Advantage Actor-Critic (Mnih et al., 2016)

A3C parallelizes training across multiple CPU workers:
- Each worker has its own environment instance and policy copy
- Workers collect experience asynchronously and compute gradients
- Gradients are applied to a shared global network
- No experience replay needed — parallelism provides diverse, decorrelated data

**Key Insight**: Asynchronous updates implicitly stabilize learning similar to experience replay in DQN, but without memory overhead. A3C achieved state-of-the-art results on Atari while training on CPUs rather than GPUs.

## 6. Implementing a Simple Policy Gradient (REINFORCE) from Scratch

Let's implement REINFORCE for the CartPole environment. We'll build everything from scratch using PyTorch to ensure deep understanding of every component.

In [ ]:
# 📦 Imports and Environment Setup
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create environment
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]  # 4
action_dim = env.action_space.n             # 2
print(f"State dimension: {state_dim}, Action dimension: {action_dim}")

In [ ]:
# 🧠 Define the Policy Network
# The policy network maps states to action probabilities using a softmax output

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits
    
    def get_action(self, state):
        """Sample an action from the policy distribution."""
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0).to(device)
            logits = self.forward(state)
            probs = torch.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_prob = dist.log_prob(action)
        return action.item(), log_prob.item()
    
    def evaluate_actions(self, states, actions):
        """Compute log probabilities for given state-action pairs."""
        logits = self.forward(states)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        log_probs = dist.log_prob(actions)
        return log_probs

# Initialize policy
policy = PolicyNetwork(state_dim, action_dim).to(device)
optimizer = optim.Adam(policy.parameters(), lr=0.01)
print("Policy network initialized!")
print(policy)

In [ ]:
# 🎮 REINFORCE Training Loop
# We collect full episodes, compute returns, and update the policy

def compute_returns(rewards, gamma=0.99):
","    """Compute discounted returns for an episode."""
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

def train_reinforce(env, policy, optimizer, num_episodes=1000, gamma=0.99, print_interval=100):
    """Train using the REINFORCE algorithm."""
    episode_rewards = []
    episode_lengths = []
    
    for episode in range(num_episodes):
        state, _ = env.reset(seed=SEED + episode)
        log_probs = []
        rewards = []
        states = []
        actions = []
        done = False
        
        # Collect trajectory
        while not done:
            action, log_prob = policy.get_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            states.append(state)
            actions.append(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        # Compute returns
        returns = compute_returns(rewards, gamma)
        returns = torch.FloatTensor(returns).to(device)
        
        # Normalize returns for stability (reduce variance)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        # Convert to tensors
        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.LongTensor(actions).to(device)
        log_probs = policy.evaluate_actions(states, actions)
        
        # Compute loss: negative because we do gradient descent on -J(θ)
        loss = -(log_probs * returns).sum()
        
        # Update policy
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Logging
        total_reward = sum(rewards)
        episode_rewards.append(total_reward)
        episode_lengths.append(len(rewards))
        
        if (episode + 1) % print_interval == 0:
            avg_reward = np.mean(episode_rewards[-print_interval:])
            print(f"Episode {episode + 1}/{num_episodes} | Avg Reward (last {print_interval}): {avg_reward:.1f} | Length: {len(rewards)}")
    
    return episode_rewards, episode_lengths

print("Starting REINFORCE training...")
rewards_reinforce, lengths_reinforce = train_reinforce(env, policy, optimizer, num_episodes=1000)
print("Training complete!")

In [ ]:
# 📈 Visualize REINFORCE Training Progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Episode Rewards
axes[0].plot(rewards_reinforce, alpha=0.3, color='blue', label='Raw')
# Moving average
window = 50
if len(rewards_reinforce) >= window:
    ma = np.convolve(rewards_reinforce, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards_reinforce)), ma, color='red', linewidth=2, label=f'MA({window})')
axes[0].axhline(y=500, color='green', linestyle='--', label='Max Reward (500)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('REINFORCE: Episode Rewards over Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Episode Lengths
axes[1].plot(lengths_reinforce, alpha=0.3, color='purple')
if len(lengths_reinforce) >= window:
    ma_len = np.convolve(lengths_reinforce, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window-1, len(lengths_reinforce)), ma_len, color='orange', linewidth=2, label=f'MA({window})')
axes[1].axhline(y=500, color='green', linestyle='--', label='Max Length (500)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Episode Length')
axes[1].set_title('REINFORCE: Episode Lengths over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final average reward (last 100 episodes): {np.mean(rewards_reinforce[-100:]):.1f}")

In [ ]:
# 🧪 Test the Learned REINFORCE Policy
# Let's watch our trained agent perform!

def evaluate_policy(env, policy, num_episodes=5, render=False):
    """Evaluate the trained policy."""
    eval_rewards = []
    for ep in range(num_episodes):
        state, _ = env.reset(seed=SEED + 1000 + ep)
        total_reward = 0
        done = False
        
        while not done:
            action, _ = policy.get_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        
        eval_rewards.append(total_reward)
        print(f"Evaluation Episode {ep + 1}: Reward = {total_reward}")
    
    print(f"\nAverage Evaluation Reward: {np.mean(eval_rewards):.1f} (+/- {np.std(eval_rewards):.1f})")
    return eval_rewards

print("Evaluating trained REINFORCE policy...")
eval_rewards = evaluate_policy(env, policy, num_episodes=10)

# Visualize action probabilities for a sample state
sample_state = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)
with torch.no_grad():
    state_tensor = torch.FloatTensor(sample_state).unsqueeze(0).to(device)
    logits = policy(state_tensor)
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

print(f"\nSample state action probabilities: Left={probs[0]:.4f}, Right={probs[1]:.4f}")

## 7. Training on CartPole or a Continuous Control Environment

Now let's implement an **Actor-Critic** agent and compare its training stability with REINFORCE. The Actor-Critic uses bootstrapping (TD learning) which typically learns faster and more stably.

In [ ]:
# 🎭 Actor-Critic Architecture
# The Actor outputs action probabilities, the Critic estimates state values

class ActorCriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(ActorCriticNetwork, self).__init__()
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # Actor head: policy
        self.actor = nn.Linear(hidden_dim, action_dim)
        # Critic head: value function
        self.critic = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features)
        return logits, value
    
    def get_action_and_value(self, state):
        """Sample action and get its value estimate."""
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0).to(device)
            logits, value = self.forward(state)
            probs = torch.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_prob = dist.log_prob(action)
        return action.item(), log_prob.item(), value.item()
    
    def evaluate(self, states, actions):
        """Evaluate actions for training."""
        logits, values = self.forward(states)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_probs, values.squeeze(-1), entropy

# Initialize Actor-Critic
ac_model = ActorCriticNetwork(state_dim, action_dim).to(device)
ac_optimizer = optim.Adam(ac_model.parameters(), lr=0.001)
print("Actor-Critic network initialized!")
print(ac_model)

In [ ]:
# 🔄 Actor-Critic Training Loop (A2C-style)
# Uses 1-step TD error for lower variance updates

def train_actor_critic(env, model, optimizer, num_episodes=1000, gamma=0.99, print_interval=100):
    """Train using Actor-Critic with TD(0) updates."""
    episode_rewards = []
    episode_lengths = []
    
    for episode in range(num_episodes):
        state, _ = env.reset(seed=SEED + episode)
        log_probs = []
        values = []
        rewards = []
        dones = []
        states = []
        actions = []
        
        done = False
        
        # Collect single episode
        while not done:
            action, log_prob, value = model.get_action_and_value(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            states.append(state)
            actions.append(action)
            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)
            dones.append(done)
            
            state = next_state
        
        # Compute returns and advantages using bootstrapping
        returns = []
        advantages = []
        G = 0
        
        # Process backwards
        for t in reversed(range(len(rewards))):
            if dones[t]:
                next_value = 0
            else:
                # Bootstrap with next state value
                with torch.no_grad():
                    next_state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                    _, next_value_tensor = model(next_state_tensor)
                    next_value = next_value_tensor.item()
            
            G = rewards[t] + gamma * G if not dones[t] else rewards[t]
            returns.insert(0, G)
            
            # Advantage = Return - Value estimate
            advantage = G - values[t]
            advantages.insert(0, advantage)
        
        # Convert to tensors
        states_t = torch.FloatTensor(np.array(states)).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        returns_t = torch.FloatTensor(returns).to(device)
        advantages_t = torch.FloatTensor(advantages).to(device)
        
        # Normalize advantages
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-9)
        
        # Evaluate
        log_probs_t, values_t, entropy = model.evaluate(states_t, actions_t)
        
        # Actor loss (policy gradient with advantage)
        actor_loss = -(log_probs_t * advantages_t).mean()
        
        # Critic loss (MSE between predicted values and returns)
        critic_loss = nn.MSELoss()(values_t, returns_t)
        
        # Entropy bonus (encourages exploration)
        entropy_bonus = entropy.mean()
        
        # Combined loss
        loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy_bonus
        
        # Update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Logging
        total_reward = sum(rewards)
        episode_rewards.append(total_reward)
        episode_lengths.append(len(rewards))
        
        if (episode + 1) % print_interval == 0:
            avg_reward = np.mean(episode_rewards[-print_interval:])
            print(f"Episode {episode + 1}/{num_episodes} | Avg Reward: {avg_reward:.1f} | "
                  f"Actor Loss: {actor_loss.item():.3f} | Critic Loss: {critic_loss.item():.3f}")
    
    return episode_rewards, episode_lengths

print("Starting Actor-Critic training...")
rewards_ac, lengths_ac = train_actor_critic(env, ac_model, ac_optimizer, num_episodes=1000)
print("Training complete!")

In [ ]:
# 📊 Compare REINFORCE vs Actor-Critic
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

window = 50

# Plot 1: Reward Comparison
axes[0].plot(rewards_reinforce, alpha=0.2, color='blue', label='REINFORCE (raw)')
axes[0].plot(rewards_ac, alpha=0.2, color='red', label='Actor-Critic (raw)')

if len(rewards_reinforce) >= window:
    ma_r = np.convolve(rewards_reinforce, np.ones(window)/window, mode='valid')
    ma_ac = np.convolve(rewards_ac, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards_reinforce)), ma_r, color='blue', linewidth=2.5, label='REINFORCE (MA50)')
    axes[0].plot(range(window-1, len(rewards_ac)), ma_ac, color='red', linewidth=2.5, label='Actor-Critic (MA50)')

axes[0].axhline(y=500, color='green', linestyle='--', alpha=0.7, label='Max (500)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('📈 REINFORCE vs Actor-Critic: Reward Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Training Stability (rolling std)
roll_window = 100
if len(rewards_reinforce) >= roll_window and len(rewards_ac) >= roll_window:
    std_r = [np.std(rewards_reinforce[max(0,i-roll_window):i+1]) for i in range(len(rewards_reinforce))]
    std_ac = [np.std(rewards_ac[max(0,i-roll_window):i+1]) for i in range(len(rewards_ac))]
    axes[1].plot(std_r, color='blue', alpha=0.7, label='REINFORCE')
    axes[1].plot(std_ac, color='red', alpha=0.7, label='Actor-Critic')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Rolling Std Dev of Reward')
    axes[1].set_title('🎯 Training Stability Comparison (Lower is Better)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"REINFORCE final 100-episode avg: {np.mean(rewards_reinforce[-100:]):.1f}")
print(f"Actor-Critic final 100-episode avg: {np.mean(rewards_ac[-100:]):.1f}")
print(f"REINFORCE variance (last 100): {np.var(rewards_reinforce[-100:]):.1f}")
print(f"Actor-Critic variance (last 100): {np.var(rewards_ac[-100:]):.1f}")

## 8. Visualizing Training Progress and Learned Policy

Let's create richer visualizations to understand how the policy evolves during training and analyze the learned behavior.

In [ ]:
# 🗺️ Policy Landscape Visualization
# Visualize how the policy responds to different state configurations

def visualize_policy_landscape(policy, env_name='CartPole-v1'):
    """Visualize action probabilities across 2D state slices."""
    test_env = gym.make(env_name)
    
    # CartPole state: [cart_position, cart_velocity, pole_angle, pole_angular_velocity]
    # We'll vary pole_angle and pole_angular_velocity while keeping others at 0
    
    angles = np.linspace(-0.3, 0.3, 50)      # pole angle
    ang_vels = np.linspace(-2.0, 2.0, 50)    # angular velocity
    
    prob_right = np.zeros((50, 50))
    
    for i, angle in enumerate(angles):
        for j, ang_vel in enumerate(ang_vels):
            state = np.array([0.0, 0.0, angle, ang_vel], dtype=np.float32)
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                logits = policy(state_t)
                probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
                prob_right[j, i] = probs[1]  # Probability of moving right
    
    plt.figure(figsize=(10, 8))
    plt.imshow(prob_right, origin='lower', extent=[-0.3, 0.3, -2.0, 2.0], aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(label='Probability of Right Action')
    plt.contour(angles, ang_vels, prob_right, levels=[0.5], colors='black', linewidths=2)
    plt.xlabel('Pole Angle (rad)')
    plt.ylabel('Pole Angular Velocity (rad/s)')
    plt.title('🎯 REINFORCE Learned Policy Landscape\n(Green = Push Right, Red = Push Left, Black Line = Decision Boundary)')
    plt.grid(True, alpha=0.3)
    plt.show()

print("Visualizing REINFORCE policy landscape...")
visualize_policy_landscape(policy)

print("\nVisualizing Actor-Critic policy landscape...")
visualize_policy_landscape(ac_model)

In [ ]:
# 📉 Learning Curve Analysis with Confidence Intervals
# Run multiple seeds to show variance across runs

def run_multiple_trials(algorithm='reinforce', num_trials=3, num_episodes=500):
    """Run multiple training trials and return all reward histories."""
    all_rewards = []
    
    for trial in range(num_trials):
        trial_seed = SEED + trial * 100
        random.seed(trial_seed)
        np.random.seed(trial_seed)
        torch.manual_seed(trial_seed)
        
        trial_env = gym.make('CartPole-v1')
        
        if algorithm == 'reinforce':
            trial_policy = PolicyNetwork(state_dim, action_dim).to(device)
            trial_opt = optim.Adam(trial_policy.parameters(), lr=0.01)
            rewards, _ = train_reinforce(trial_env, trial_policy, trial_opt, num_episodes=num_episodes, print_interval=9999)
        else:
            trial_model = ActorCriticNetwork(state_dim, action_dim).to(device)
            trial_opt = optim.Adam(trial_model.parameters(), lr=0.001)
            rewards, _ = train_actor_critic(trial_env, trial_model, trial_opt, num_episodes=num_episodes, print_interval=9999)
        
        all_rewards.append(rewards)
        trial_env.close()
    
    return np.array(all_rewards)

print("Running multiple trials for statistical comparison...")
print("This may take a few minutes...")
rewards_r_multi = run_multiple_trials('reinforce', num_trials=3, num_episodes=500)
rewards_ac_multi = run_multiple_trials('actor_critic', num_trials=3, num_episodes=500)

# Plot with confidence intervals
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(500)
mean_r = rewards_r_multi.mean(axis=0)
std_r = rewards_r_multi.std(axis=0)
mean_ac = rewards_ac_multi.mean(axis=0)
std_ac = rewards_ac_multi.std(axis=0)

ax.plot(x, mean_r, color='blue', linewidth=2, label='REINFORCE (mean)')
ax.fill_between(x, mean_r - std_r, mean_r + std_r, color='blue', alpha=0.2)
ax.plot(x, mean_ac, color='red', linewidth=2, label='Actor-Critic (mean)')
ax.fill_between(x, mean_ac - std_ac, mean_ac + std_ac, color='red', alpha=0.2)

ax.axhline(y=500, color='green', linestyle='--', alpha=0.7, label='Max Reward')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('📊 Multi-Seed Comparison: Mean ± Std Dev (3 trials each)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("Multi-seed comparison complete!")

## 9. Pros, Cons, and When to Use Policy Gradient vs DQN

### ⚖️ Comprehensive Comparison

| Aspect | DQN (Value-Based) | REINFORCE (Policy-Based) | Actor-Critic |
|--------|-------------------|--------------------------|--------------|
| **Action Space** | Discrete only | Discrete & Continuous | Discrete & Continuous |
| **Sample Efficiency** | High (replay buffer) | Low (on-policy) | Medium |
| **Training Stability** | Medium (target networks help) | Low (high variance) | High (bootstrapping) |
| **Convergence** | Stable but can oscillate | Smooth convergence | Smooth & fast |
| **Exploration** | ε-greedy (external) | Stochastic policy (intrinsic) | Stochastic policy + entropy bonus |
| **Memory** | High (replay buffer) | Low (episodic) | Low |
| **Parallelization** | Difficult | Easy | Very Easy (A3C) |
| **Best For** | Discrete games (Atari) | Simple environments, theory | Complex continuous control |

### 🎯 Decision Framework

**Choose DQN when:**
- Actions are discrete and finite
- Sample efficiency is critical
- You have sufficient memory for experience replay
- The environment is deterministic or near-deterministic

**Choose Policy Gradient when:**
- Actions are continuous (robotics, motor control)
- You need a stochastic policy (games with hidden information)
- The action space is too large for value-based methods
- You want direct policy optimization

**Choose Actor-Critic when:**
- You need the best of both worlds
- Training stability is important
- You have access to parallel environments (A3C/A2C)
- You're working on state-of-the-art continuous control (PPO, SAC)

## 10. Modern Actor-Critic Algorithms (PPO, SAC – brief overview)

The Actor-Critic framework has spawned the most successful modern RL algorithms:

### 🚀 Proximal Policy Optimization (PPO) — Schulman et al., 2017

**Problem**: Policy gradient methods can make destructively large policy updates, causing performance collapse.

**Solution**: PPO clips the probability ratio to prevent overly aggressive updates:

$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)A_t\right)\right]$$

Where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the probability ratio.

**Why PPO dominates**: Simple to implement, robust hyperparameters, state-of-the-art on continuous control benchmarks, widely adopted in industry.

### 🧊 Soft Actor-Critic (SAC) — Haarnoja et al., 2018

**Problem**: Standard Actor-Critic can converge to suboptimal deterministic policies and lacks exploration incentives.

**Solution**: SAC maximizes **entropy-regularized** reward:

$$J(\pi) = \sum_t \mathbb{E}_{(s_t, a_t) \sim \rho_\pi}\left[r(s_t, a_t) + \alpha \mathcal{H}(\pi(\cdot|s_t))\right]$$

**Key Features**:
- **Off-policy**: Can reuse past experience (high sample efficiency)
- **Maximum Entropy**: Encourages exploration naturally
- **Twin Q-Networks**: Mitigates overestimation bias
- **Automatic Temperature Tuning**: Adapts exploration-exploitation trade-off

**Results**: SAC achieves state-of-the-art sample efficiency on MuJoCo benchmarks and is the go-to algorithm for real-world robotic learning.

### 📋 Algorithm Evolution Timeline

```
REINFORCE (1992) → Actor-Critic (2000s) → A3C (2016) → A2C (2016) → PPO (2017) → SAC (2018)
         ↑                    ↑                ↑              ↑            ↑            ↑
    High Variance       Bootstrapping     Parallelism    Synchronous    Clipping    Entropy + Off-policy
```

## 🛠️ Hands-On Exercises

Complete these exercises to solidify your understanding of policy gradient methods. Attempt each before checking the solutions!

### Exercise 1: Implement REINFORCE from Scratch
Implement the REINFORCE algorithm without using the provided code. Create your own PolicyNetwork class and training loop. Test it on CartPole-v1.

In [ ]:
# ✏️ Exercise 1: Your REINFORCE Implementation
# Hints:
# - Create a neural network with input=4, hidden=128, output=2
# - Use torch.distributions.Categorical for action sampling
# - Compute discounted returns manually (no built-in functions)
# - Update using policy gradient loss

import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np

# Your code here:



### Exercise 2: Train on CartPole and Analyze Learning Curves
Train your REINFORCE agent for 1000 episodes. Plot the raw rewards and a 50-episode moving average. Identify where learning "takes off" (the episode where performance starts consistently improving).

In [ ]:
# ✏️ Exercise 2: Training and Analysis
# Train your agent and create plots showing:
# 1. Raw episode rewards
# 2. 50-episode moving average
# 3. A vertical line at the "take-off" episode

# Your code here:



### Exercise 3: Add Baseline (Advantage) to Reduce Variance
Modify REINFORCE to subtract a learned state-value baseline. Implement a separate value network (or shared network) that estimates V(s). Use the TD error (or Monte Carlo return minus baseline) as the advantage for policy updates.

In [ ]:
# ✏️ Exercise 3: REINFORCE with Baseline
# Implement a value baseline network:
# - V(s) estimates expected return from state s
# - Advantage = G_t - V(s_t)
# - Update value network to minimize MSE with returns
# - Use advantage instead of raw returns for policy gradient

# Your code here:



### Exercise 4: Build a Simple Actor-Critic Agent
Implement the full Actor-Critic algorithm with separate actor and critic networks. Use bootstrapped TD errors (1-step) for the critic update and advantage estimates for the actor. Train on CartPole and compare with your REINFORCE results.

In [ ]:
# ✏️ Exercise 4: Actor-Critic Implementation
# Requirements:
# - Separate Actor (policy) and Critic (value) networks
# - 1-step TD error: δ = r + γV(s') - V(s)
# - Actor loss: -log_prob * δ.detach()
# - Critic loss: MSE between V(s) and target (r + γV(s'))
# - Add entropy bonus for exploration

# Your code here:



### Exercise 5: Experiment with Different Reward Normalization Techniques
Compare three approaches:
1. No normalization (raw returns)
2. Standard score normalization: (returns - mean) / std
3. Running mean/std normalization (maintain statistics across episodes)
Train with each and compare variance and convergence speed.

In [ ]:
# ✏️ Exercise 5: Reward Normalization Comparison
# Implement three training runs with different normalization strategies
# Track and compare:
# - Episode rewards
# - Policy gradient variance
# - Time to convergence

# Your code here:



### Exercise 6: Compare Policy Gradient Methods with DQN on the Same Environment
Implement or use a pre-built DQN agent. Train both DQN and your Actor-Critic on CartPole with identical network sizes and training durations. Compare:
- Sample efficiency (episodes to solve)
- Final performance
- Training stability (variance of rewards)

In [ ]:
# ✏️ Exercise 6: DQN vs Actor-Critic Comparison
# You can use a simple DQN implementation or reference Day 73's code
# Ensure fair comparison:
# - Same network architecture (2 hidden layers, 128 units)
# - Same learning rate (or tuned individually)
# - Same number of total training steps

# Your code here:



### Exercise 7: Implement Entropy Regularization
Add an entropy bonus to your policy gradient updates. The entropy of a categorical distribution is:
$$H(\pi) = -\sum_a \pi(a|s) \log \pi(a|s)$$
Experiment with different entropy coefficients (0.001, 0.01, 0.1) and observe the effect on exploration and final performance.

In [ ]:
# ✏️ Exercise 7: Entropy Regularization
# Modify your policy gradient to include:
# loss = -log_prob * advantage - entropy_coef * entropy
# Try entropy_coef in [0.0, 0.001, 0.01, 0.1]
# Plot comparison of exploration (action entropy over time) and performance

# Your code here:



### Exercise 8: Visualize Policy Evolution During Training
Save snapshots of your policy network every 100 episodes during training. Visualize how the action probabilities (or the decision boundary) change over time for a grid of states. Create an animated or static multi-panel figure showing policy evolution.

In [ ]:
# ✏️ Exercise 8: Policy Evolution Visualization
# During training, periodically evaluate policy on a grid of states
# For CartPole: vary pole_angle and pole_angular_velocity
# Create a figure with subplots showing policy at episode 0, 250, 500, 750, 1000

# Your code here:



### Exercise 9: Discuss Challenges in High-Dimensional Action Spaces
Write a brief analysis (2-3 paragraphs) addressing:
1. Why REINFORCE struggles with high-dimensional continuous actions
2. How Actor-Critic methods (especially with Gaussian policies) handle continuous actions
3. Why modern algorithms like PPO and SAC use more sophisticated distribution parameterizations (e.g., state-dependent standard deviations, normalizing flows)
4. The curse of dimensionality in action spaces and how entropy regularization helps

In [ ]:
# ✏️ Exercise 9: Written Analysis
# Write your analysis as a markdown string and display it

analysis = """
# Analysis: Challenges in High-Dimensional Action Spaces

[Your analysis here...]
"""

print(analysis)


## Solutions (Review After Attempting)

Below are detailed solutions and explanations for each exercise. Study them after attempting the problems yourself!

### ✅ Solution 1: REINFORCE from Scratch

```python
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import numpy as np

class Policy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
    
    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)
    
    def act(self, state):
        state = torch.FloatTensor(state).unsqueeze(0)
        probs = self.forward(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

def train():
    env = gym.make('CartPole-v1')
    policy = Policy(4, 2)
    optimizer = optim.Adam(policy.parameters(), lr=0.01)
    gamma = 0.99
    
    for episode in range(1000):
        state, _ = env.reset()
        log_probs = []
        rewards = []
        done = False
        
        while not done:
            action, log_prob = policy.act(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
        
        # Compute returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        # Policy gradient update
        loss = []
        for log_prob, G in zip(log_probs, returns):
            loss.append(-log_prob * G)
        
        optimizer.zero_grad()
        loss = torch.stack(loss).sum()
        loss.backward()
        optimizer.step()
        
        if episode % 100 == 0:
            print(f"Episode {episode}, Reward: {sum(rewards)}")

train()
```

**Key Points**:
- The policy network outputs action probabilities via softmax
- We sample actions using `Categorical` distribution
- Returns are computed backwards (efficient O(n) computation)
- Normalizing returns reduces variance significantly
- The loss is the negative log-probability weighted by return (gradient ascent on expected reward)

### ✅ Solution 2: Learning Curve Analysis

```python
import matplotlib.pyplot as plt

# After training, rewards_history contains episode rewards
rewards_history = []  # Fill with your training rewards

# Compute moving average
window = 50
moving_avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')

# Find take-off point (where MA first exceeds threshold consistently)
threshold = 400
take_off = None
for i in range(len(moving_avg)):
    if np.all(moving_avg[i:i+20] > threshold):
        take_off = i + window - 1
        break

plt.figure(figsize=(10, 6))
plt.plot(rewards_history, alpha=0.3, label='Raw')
plt.plot(range(window-1, len(rewards_history)), moving_avg, label='MA(50)', linewidth=2)
if take_off:
    plt.axvline(take_off, color='red', linestyle='--', label=f'Take-off: Ep {take_off}')
plt.axhline(500, color='green', linestyle='--', alpha=0.5, label='Max')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('REINFORCE Learning Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

**Analysis Tips**:
- The "take-off" point indicates when the policy has learned a reasonably good state representation
- Early episodes show high variance as the policy is nearly random
- After take-off, the moving average should trend upward smoothly
- If no clear take-off occurs, try increasing learning rate or network capacity

### ✅ Solution 3: REINFORCE with Baseline

```python
class ValueNetwork(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_with_baseline():
    policy = Policy(4, 2)
    value_net = ValueNetwork(4)
    policy_opt = optim.Adam(policy.parameters(), lr=0.01)
    value_opt = optim.Adam(value_net.parameters(), lr=0.01)
    
    for episode in range(1000):
        state, _ = env.reset()
        log_probs = []
        rewards = []
        states = []
        done = False
        
        while not done:
            action, log_prob = policy.act(state)
            states.append(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
        
        # Compute returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        states = torch.FloatTensor(np.array(states))
        
        # Baseline values
        values = value_net(states)
        advantages = returns - values.detach()
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-9)
        
        # Policy update
        policy_loss = sum([-lp * adv for lp, adv in zip(log_probs, advantages)])
        policy_opt.zero_grad()
        policy_loss.backward()
        policy_opt.step()
        
        # Value update
        value_loss = nn.MSELoss()(values, returns)
        value_opt.zero_grad()
        value_loss.backward()
        value_opt.step()
```

**Why Baselines Help**:
- The baseline $V(s)$ doesn't change the expected gradient (unbiased)
- It reduces variance by centering updates around zero
- Good actions (advantage > 0) increase probability, bad actions decrease it
- The value network learns simultaneously, bootstrapping knowledge across episodes

### ✅ Solution 4: Full Actor-Critic

```python
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU()
        )
        self.actor = nn.Linear(128, action_dim)
        self.critic = nn.Linear(128, 1)
    
    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features)
        return torch.softmax(logits, dim=-1), value.squeeze(-1)
    
    def get_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0)
        probs, value = self.forward(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action), value

def train_actor_critic():
    model = ActorCritic(4, 2)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    gamma = 0.99
    
    for episode in range(1000):
        state, _ = env.reset()
        log_probs = []
        values = []
        rewards = []
        done = False
        
        while not done:
            action, log_prob, value = model.get_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)
            state = next_state
        
        # Compute returns and advantages
        returns = []
        advantages = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        
        returns = torch.tensor(returns)
        values = torch.stack(values).squeeze()
        log_probs = torch.stack(log_probs)
        
        advantages = returns - values.detach()
        
        # Losses
        actor_loss = -(log_probs * advantages).mean()
        critic_loss = nn.MSELoss()(values, returns)
        entropy = -(torch.exp(log_probs) * log_probs).mean()  # Approximate
        
        loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

**Architecture Decisions**:
- Shared layers allow the actor and critic to learn complementary features
- Separate heads prevent one task from dominating the gradient
- TD(0) provides lower variance than Monte Carlo but introduces bias
- The entropy term prevents premature convergence to deterministic policies

### ✅ Solution 5: Reward Normalization

```python
class RunningNormalizer:
    """Online normalization using running statistics."""
    def __init__(self):
        self.mean = 0
        self.var = 1
        self.count = 0
    
    def update(self, x):
        self.count += 1
        delta = x - self.mean
        self.mean += delta / self.count
        self.var += delta * (x - self.mean)
    
    def normalize(self, x):
        if self.count > 1:
            std = np.sqrt(self.var / (self.count - 1))
            return (x - self.mean) / (std + 1e-8)
        return x

# Usage in training loop:
normalizer = RunningNormalizer()
returns = compute_returns(rewards)
for g in returns:
    normalizer.update(g)
normalized_returns = [normalizer.normalize(g) for g in returns]
```

**Comparison Results** (typical):
- **No normalization**: High variance, slow convergence, unstable late training
- **Batch normalization**: Good variance reduction, but sensitive to outlier episodes
- **Running normalization**: Most stable, adapts to changing reward scales, best for long training

### ✅ Solution 6: DQN vs Actor-Critic

**Key Differences in Implementation**:
- DQN uses a replay buffer (minibatch updates) while Actor-Critic is typically on-policy
- DQN learns Q(s,a) values; Actor-Critic learns both π(a|s) and V(s)
- DQN uses target networks for stability; Actor-Critic can use bootstrapping

**Expected Results on CartPole**:
- DQN usually solves in 200-400 episodes with tuned hyperparameters
- Actor-Critic often solves in 300-600 episodes but with smoother learning curves
- DQN shows more sample efficiency (reuses experiences)
- Actor-Critic shows better stability in continuous action spaces

**Fair Comparison Setup**:
```python
# Ensure identical network capacity
dqn_network = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 2))
ac_shared = nn.Sequential(nn.Linear(4, 128), nn.ReLU())
# Both have ~ same parameters

# Track episodes to reach reward > 475 for 10 consecutive episodes
```

### ✅ Solution 7: Entropy Regularization

```python
def compute_entropy(probs):
    """Compute entropy of categorical distribution."""
    return -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean()

# In training loop:
probs, values = model(states)
entropy = compute_entropy(probs)
loss = actor_loss + 0.5 * critic_loss - entropy_coef * entropy

# Experiment results:
# coef=0.0: Fast initial learning but may plateau early (insufficient exploration)
# coef=0.01: Balanced exploration-exploitation, best final performance
# coef=0.1: Too much exploration, slow convergence, noisy policy
```

**Entropy Scheduling**:
For best results, decay the entropy coefficient over time:
```python
entropy_coef = max(0.001, initial_coef * (1 - episode / total_episodes))
```

### ✅ Solution 8: Policy Evolution Visualization

```python
def visualize_evolution(policies, env_name='CartPole-v1'):
    """Visualize policy evolution at different training stages."""
    angles = np.linspace(-0.3, 0.3, 30)
    ang_vels = np.linspace(-2.0, 2.0, 30)
    
    fig, axes = plt.subplots(1, len(policies), figsize=(20, 4))
    
    for idx, (policy, episode) in enumerate(policies):
        prob_right = np.zeros((30, 30))
        for i, angle in enumerate(angles):
            for j, ang_vel in enumerate(ang_vels):
                state = np.array([0.0, 0.0, angle, ang_vel], dtype=np.float32)
                with torch.no_grad():
                    state_t = torch.FloatTensor(state).unsqueeze(0)
                    logits = policy(state_t)
                    probs = torch.softmax(logits, dim=-1).numpy()[0]
                    prob_right[j, i] = probs[1]
        
        im = axes[idx].imshow(prob_right, origin='lower', 
                              extent=[-0.3, 0.3, -2.0, 2.0], 
                              aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
        axes[idx].set_title(f'Episode {episode}')
        axes[idx].set_xlabel('Pole Angle')
        if idx == 0:
            axes[idx].set_ylabel('Angular Velocity')
    
    plt.colorbar(im, ax=axes, label='P(Right)')
    plt.suptitle('Policy Evolution During Training')
    plt.show()

# Save snapshots during training:
# if episode % 200 == 0:
#     policies.append((copy.deepcopy(policy), episode))
```

**Expected Evolution**:
- Episode 0: Nearly uniform (random policy, ~50% everywhere)
- Episode 200: Emerging structure, some regions favor left/right
- Episode 500: Clear decision boundary, but slightly fuzzy
- Episode 1000: Sharp boundary, confident predictions (near 0 or 1)

### ✅ Solution 9: High-Dimensional Action Spaces Analysis

**Sample Answer**:

REINFORCE struggles with high-dimensional continuous action spaces primarily because it relies on Monte Carlo sampling of entire trajectories. In continuous spaces, the probability of sampling any specific action is zero, requiring parameterized distributions (typically Gaussian). The gradient variance scales poorly with action dimension — each additional action dimension adds noise to the gradient estimates. Without a baseline, this variance becomes prohibitive, leading to extremely slow or unstable learning.

Actor-Critic methods handle continuous actions by parameterizing the policy as a conditional probability distribution, most commonly a Gaussian where the mean and standard deviation are neural network outputs: $a \sim \mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$. The Critic's value estimates provide lower-variance advantages, making learning feasible even in 10-50 dimensional action spaces (typical in robotics). However, a fixed Gaussian may be too restrictive for complex, multimodal action distributions.

Modern algorithms like PPO and SAC address these limitations through sophisticated distribution modeling. PPO maintains stable updates via clipped objectives, allowing larger networks that can model complex state-dependent covariances. SAC uses maximum entropy principles to naturally encourage exploration without manual tuning. Advanced methods employ normalizing flows or mixture models to represent arbitrary action distributions. Entropy regularization is crucial — without it, policies collapse to deterministic actions early in training, getting stuck in local optima. The entropy bonus ensures sufficient exploration to discover better action modes even in high-dimensional spaces where random exploration is exponentially unlikely to succeed.

---

## 🎓 Day 74 Complete!

You have now understood the core ideas behind policy-based and actor-critic reinforcement learning — the foundation of modern RL algorithms. From the elegant simplicity of REINFORCE to the powerful synergy of Actor-Critic methods, you've built the mathematical intuition and practical skills that underpin state-of-the-art systems.

### 🔮 Tomorrow: Proximal Policy Optimization (PPO) and Modern RL

On Day 75, we dive into **Proximal Policy Optimization (PPO)** — the algorithm that has become the default choice for most RL practitioners. We'll explore its clipped surrogate objective, implementation details, and why it strikes the perfect balance between simplicity and performance. We'll also touch on **Soft Actor-Critic (SAC)** for off-policy continuous control and discuss how these algorithms power real-world applications from robotic manipulation to large language model fine-tuning (RLHF).

Keep pushing forward — you're mastering the algorithms that are shaping the future of AI! 💪🤖

---
**Python & AI Learning Path | Day 74 / 369** ⭐
*"The best way to predict the future is to implement it."*